# TP 0 — Découvrir Spark, SQL et Parquet
**Durée indicative : 35–50 min**

Objectifs : lire un CSV, inspecter un schéma, filtrer en SQL, écrire et relire du Parquet.

> **Environment:** Databricks notebook in a Git folder. Attach a compute resource with PySpark available. Data files are in the adjacent `data/` folder. This lab uses only **synthetic** data.

> **Expected skill level:** first experience with Big Data / Spark. Follow the cells in order.


## 1. Charger les données
Le CSV de démonstration représente 4 employés fictifs. À votre avis, `Salary` est-il une chaîne ou un nombre après une lecture CSV ?

In [ ]:
from pathlib import Path
import csv
from pyspark.sql import functions as F

# Databricks Runtime 14+ sets the current directory to the notebook's folder.
DATA_DIR = Path.cwd().parent / "data"
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}. Open the notebook from the Git folder."

def read_small_csv(filename):
    # Python reads the small Git-tracked CSV on the driver; Spark does the processing.
    # For big data in production, read directly from cloud storage instead.
    with (DATA_DIR / filename).open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    return spark.createDataFrame(rows)


In [ ]:
df = read_small_csv("employes_demo.csv")
df.show()
df.printSchema()

## 2. Corriger le type et poser une question en SQL
**Question :** quels employés gagnent strictement plus de 30 000 ? Pourquoi le cast est-il utile ?

In [ ]:
df = df.withColumn("Salary", F.col("Salary").cast("integer"))
df.createOrReplaceTempView("employes")
spark.sql("SELECT FirstName, Country, Salary FROM employes WHERE Salary > 30000").show()

## 3. À vous de jouer
Afficher le nombre de collaborateurs par pays avec SQL (`GROUP BY`).

In [ ]:
# TODO: compléter la requête SQL
spark.sql("SELECT Country, COUNT(*) AS nb FROM employes GROUP BY Country").show()

## 4. Écrire et relire du Parquet
⚠️ Choisissez un sous-dossier personnel dans le volume. Ne jamais écraser la racine d’un volume partagé.

In [ ]:
# IMPORTANT: configure one writable Unity Catalog Volume before this step.
# Use a unique subdirectory to avoid overwriting a classmate's output.
OUTPUT_ROOT = "/Volumes/workspace/default/episen_volume/CHANGE_ME"
assert "CHANGE_ME" not in OUTPUT_ROOT, "Replace CHANGE_ME with your personal identifier."


In [ ]:
parquet_path = OUTPUT_ROOT.rstrip("/") + "/tp00_employes_parquet"
df.write.mode("overwrite").parquet(parquet_path)
df2 = spark.read.parquet(parquet_path)
df2.printSchema()
df2.show()

### Bilan
- Quelle différence entre **CSV** et **Parquet** ?
- Qu’est-ce qu’une **vue temporaire SQL** ?
- Dans un DataFrame, que signifie un **type** ?